# 11 - Prior Sweep Under Uniform Users

This notebook visualizes `experiments/prior_sweep_uniform.py` outputs. It asks how the Gaussian prior covariance scale affects policy performance when the true latent vectors are sampled from a near-uniform population.

The swept prior is:

$$\Sigma_0 = s I$$

where `s` is `prior_scale`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError("Could not find project root containing src/ and experiments/.")


REPO_ROOT = find_project_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "experiments" / "results"

for path in [REPO_ROOT, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

from _sweep_helpers import POLICY_STYLE, PALETTE, setup, style_ax

setup()
print(f"Repo root: {REPO_ROOT}")

## Load Prior Sweep

Run the experiment first if no file is found:

```powershell
python -m experiments.prior_sweep_uniform --values 0.25 0.5 1 2 4 9 --dim 6 --horizon 20 --users 500 --no-myopic
```

Leave `SWEEP_FILE = None` to load the newest prior-sweep result.

In [ ]:
SWEEP_FILE = None
# SWEEP_FILE = "20260429_180240_prior_sweep_uniform_dim2_h2_p10_n6.json"


def list_prior_sweeps(limit: int = 12) -> list[Path]:
    files = sorted(
        RESULTS_DIR.glob("*prior_sweep_uniform*.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not files:
        print("No prior-sweep result files found.")
        return []
    for index, path in enumerate(files[:limit]):
        print(f"{index:>2}: {path.name}")
    return files


available_sweeps = list_prior_sweeps()

if SWEEP_FILE is None:
    if not available_sweeps:
        raise FileNotFoundError("Run experiments.prior_sweep_uniform first.")
    sweep_path = available_sweeps[0]
else:
    sweep_path = Path(SWEEP_FILE)
    if not sweep_path.is_absolute():
        sweep_path = RESULTS_DIR / sweep_path

sweep = json.loads(sweep_path.read_text(encoding="utf-8"))
config = sweep["fixed_config"]
policies = list(sweep["conditions"][0]["policies"].keys())

rows = []
for condition in sweep["conditions"]:
    for policy, metrics in condition["policies"].items():
        rows.append({"prior_scale": condition["prior_scale"], "policy": policy, **metrics})

summary = pd.DataFrame(rows)

print(f"Loaded: {sweep_path.name}")
print(f"Policies: {', '.join(policies)}")
summary.head()

## Configuration

In [ ]:
pd.DataFrame([{"parameter": key, "value": value} for key, value in config.items()])

## Summary Table

These are the final carried metrics from each prior scale. Lower estimation error and lower uncertainty are better, but they do not always move together.

In [ ]:
summary.sort_values(["prior_scale", "mean_l2_error"])[
    [
        "prior_scale",
        "policy",
        "dropout_rate",
        "mean_n_answered",
        "mean_l2_error",
        "rmse",
        "mean_d_error",
        "mean_trace",
        "sensitive_rate",
    ]
]

## Final Error And Uncertainty Versus Prior Scale

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

panels = [
    ("mean_l2_error", "Mean L2 error", "distance to true theta"),
    ("mean_d_error", "Mean D-error", "posterior uncertainty"),
    ("mean_trace", "Mean trace", "sum of marginal variances"),
]

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = summary[summary["policy"] == policy].sort_values("prior_scale")
    for ax, (metric, title, ylabel) in zip(axes, panels):
        ax.plot(
            sub["prior_scale"],
            sub[metric],
            color=style.get("color"),
            ls=style.get("ls", "-"),
            marker=style.get("marker", "o"),
            linewidth=1.8,
            markersize=4,
            label=style.get("label", policy),
        )
        ax.set_xscale("log")
        ax.set_xlabel("Prior scale")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        style_ax(ax, grid_axis="y")

axes[-1].legend(fontsize=8)
fig.suptitle("Final carried performance under different priors", y=1.03)
plt.tight_layout()
plt.show()

## Dropout And Question Exposure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

panels = [
    ("dropout_rate", "Dropout rate", "dropout rate (%)", 100.0),
    ("mean_n_answered", "Mean answered", "answered questions", 1.0),
    ("sensitive_rate", "Sensitive rate", "sensitive / asked (%)", 100.0),
]

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = summary[summary["policy"] == policy].sort_values("prior_scale")
    for ax, (metric, title, ylabel, scale) in zip(axes, panels):
        ax.plot(
            sub["prior_scale"],
            sub[metric] * scale,
            color=style.get("color"),
            ls=style.get("ls", "-"),
            marker=style.get("marker", "o"),
            linewidth=1.8,
            markersize=4,
            label=style.get("label", policy),
        )
        ax.set_xscale("log")
        ax.set_xlabel("Prior scale")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        style_ax(ax, grid_axis="y")

axes[-1].legend(fontsize=8)
fig.suptitle("Engagement outcomes under different priors", y=1.03)
plt.tight_layout()
plt.show()

## Accuracy Versus Uncertainty

Each point is one policy at one prior scale. Lower-left is better.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = summary[summary["policy"] == policy].sort_values("prior_scale")
    ax.plot(
        sub["mean_d_error"],
        sub["mean_l2_error"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.4,
        markersize=5,
        label=style.get("label", policy),
    )
    for _, row in sub.iterrows():
        ax.annotate(
            f"{row['prior_scale']:g}",
            (row["mean_d_error"], row["mean_l2_error"]),
            xytext=(4, 3),
            textcoords="offset points",
            fontsize=7,
            color=style.get("color"),
        )

ax.set_xlabel("Final mean D-error")
ax.set_ylabel("Final mean L2 error")
ax.set_title("Final accuracy vs uncertainty")
style_ax(ax, grid_axis="both")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Best Prior Scale Per Policy

In [ ]:
best_by_l2 = summary.loc[summary.groupby("policy")["mean_l2_error"].idxmin()]
best_by_uncertainty = summary.loc[summary.groupby("policy")["mean_d_error"].idxmin()]

best = best_by_l2[["policy", "prior_scale", "mean_l2_error", "rmse"]].rename(
    columns={"prior_scale": "best_prior_for_l2"}
).merge(
    best_by_uncertainty[["policy", "prior_scale", "mean_d_error", "mean_trace"]].rename(
        columns={"prior_scale": "best_prior_for_d_error"}
    ),
    on="policy",
)
best

## Inspect Time Curves For One Prior

The prior-sweep summary stores the names of the individual uniform-learning runs. This section loads one of them so you can inspect the full time-step curves.

In [ ]:
SELECTED_PRIOR_SCALE = sweep["prior_values"][len(sweep["prior_values"]) // 2]

condition = min(
    sweep["conditions"],
    key=lambda item: abs(float(item["prior_scale"]) - float(SELECTED_PRIOR_SCALE)),
)
run_path = RESULTS_DIR / condition["run_file"]
run_data = json.loads(run_path.read_text(encoding="utf-8"))
curves = pd.DataFrame(run_data["curves"])

print(f"Selected prior_scale: {condition['prior_scale']}")
print(f"Loaded run: {run_path.name}")
curves.head()

In [ ]:
MODE = "carried"

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = curves[(curves["mode"] == MODE) & (curves["policy"] == policy)]
    axes[0].plot(
        sub["answered"],
        sub["mean_l2_error"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=style.get("label", policy),
    )
    axes[1].plot(
        sub["answered"],
        sub["mean_d_error"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=style.get("label", policy),
    )

axes[0].set_title("Mean L2 error")
axes[0].set_ylabel("distance to true theta")
axes[1].set_title("Mean D-error")
axes[1].set_ylabel("posterior uncertainty")

for ax in axes:
    ax.set_xlabel("Answered questions")
    style_ax(ax, grid_axis="y")

axes[1].legend(fontsize=8)
fig.suptitle(f"Learning curves at prior_scale={condition['prior_scale']:g} ({MODE})", y=1.03)
plt.tight_layout()
plt.show()